# Simple agent
this is a simple agent (from AI Agents from Scratch: Single Agents)

In [11]:
import ollama
llm = "qwen3"

In [12]:
# testing the LLM 
stream = ollama.generate(model=llm, prompt='what time is it?', stream=True) 
for chunk in stream: 
    print(chunk['response'], end='', flush=True)


I don't have access to real-time data or the ability to check the current time. You may want to check your device's clock or an online time source for the current time in your time zone. Let me know if you need help with anything else!

In [5]:
# install dependencies for web search agent
%pip install -U langchain-community ddgs primp

Note: you may need to restart the kernel to use updated packages.


In [13]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

def search_web(query:str) -> str:
    wrapper = DuckDuckGoSearchAPIWrapper(max_results=3) 
    
    return DuckDuckGoSearchResults(api_wrapper=wrapper, backend="news").invoke(query)

In [21]:
# Ollama requires a tool to be registered as a dictionary
tool_search_web = {
    'type': 'function', 
    'function': {
        'name': 'search_web', 
        'description': 'Search the web', 
        'parameters': {
            'type': 'object', 
            'required':['query'], 
            'properties': {
                'query': {'type':'str', 'description': 'topic or subject to search on the web'}
            }
        }
    }
}

In [15]:
# test
search_web("most recent iphone released")

"snippet: Nvidia stock is not for the faint-hearted., title: Is Nvidia Stock a Buy in September 2026?, link: https://finance.yahoo.com/markets/stocks/articles/nvidia-stock-buy-september-2026-183500073.html, date: 2026-09-15T18:12:44+00:00, source: Motley Fool · via Yahoo Finance, snippet: Nvidia may have won the first phase of the AI revolution, but the next phase is just beginning to..., title: Nvidia May Have Won the AI Training Race, but the Bigger Opportunity Is Likely..., link: https://finance.yahoo.com/technology/ai/articles/nvidia-may-won-ai-training-215900389.html, date: 2026-09-15T22:12:44+00:00, source: Motley Fool · via Yahoo Finance, snippet: By acquiring Hugging Face, Nvidia is working to capture the talent funnel., title: Nvidia Bought Hugging Face for $12.9 Billion. This Software Acquisition Changes..., link: https://finance.yahoo.com/technology/ai/articles/nvidia-bought-hugging-face-12-213200678.html, date: 2026-09-15T21:12:44+00:00, source: Motley Fool · via Yahoo Fina

In [19]:
def search_yf(query: str) -> str: 
    wrapper = DuckDuckGoSearchAPIWrapper(max_results=3) 

    return DuckDuckGoSearchResults(api_wrapper=wrapper, backend="news").invoke(f"site:finance.yahoo.com {query}")
tool_search_yf = {
    'type': 'function', 
    'function': {
        'name': 'search_yf', 
        'description': 'Search for financial news', 
        'parameters': {
            'type': 'object', 
            'required':['query'], 
            'properties': {
                'query': {'type':'str', 'description': 'financial topic or subject to search'}
            }
        }
    }
}
search_web("nvidia")

'snippet: President Donald Trump called Nvidia CEO Jensen Huang during a company all-hands meeting and gave another full-throated ..., title: Trump calls Nvidia CEO to talk AI — and gets put on speakerphone in all-hands meeting, link: https://www.msn.com/en-us/news/other/trump-calls-nvidia-ceo-to-talk-ai-and-gets-put-on-speakerphone-in-all-hands-meeting/ar-AA2cdtGI?ocid=BingNewsVerp, date: 2026-09-14T23:13:43+00:00, source: NBC News on MSN, snippet: Now both companies are tied together through one of Elon Musk ’s more ambitious AI bets. SpaceXAI chose Nvidia’s Vera Rubin ..., title: Elon Musk sends strong signal for SpaceX, Nvidia stocks, link: https://www.thestreet.com/investing/stocks/elon-musk-spacex-nvidia-ai-2027, date: 2026-09-14T23:13:43+00:00, source: TheStreet, snippet: Nvidia CEO Jensen Huang doubles down on his 2030 AI market bet as strong earnings, supply constraints and CoreWeave raise new questions., title: Nvidia CEO Jensen Huang doubles down on his big 2030 market bet, 

# Simple Agent
Building simple agent that chooses between these two tools

In [17]:
prompt='''You are an assistant who has access to tools. It is your job to decide and invoke the tool that is most relevant to answer a query'''
messages= [{'role':'system', 'content':'prompt'}]
dic_tools = {'search_web':search_web, 'search_yf':search_yf} 


In [25]:

# convo loop 
while True: 
    try:
        q = input('you->')
    except EOFError: 
        break
    if q == 'quit' or q == 'q' or q == 'exit':
        break
    if q.strip() == "":
        continue 
    messages.append({"role":"user", 'content':q})

    # model 
    agent_res = ollama.chat(
        model=llm,
        tools=[tool_search_web, tool_search_yf],
        messages=messages
    )

    # response 
    messages.append(agent_res['message'])
    tool_calls = getattr(agent_res['message'], 'tool_calls', None)
    # run any tools the model wants to run 
    if tool_calls:
        for tool in tool_calls:
            # TODO 1: pull out the tool name and arguments from `tool`
            # hint: it's nested under tool['function'][...]
            t_name = tool['function']['name']
            t_args = tool['function']['arguments']

            # TODO 2: look up the function in dic_tools
            # what do you do if t_name isn't in dic_tools?
            f = dic_tools.get(t_name)
            if f: 
                try:
                    # TODO 3: call the function with t_args, catch exceptions
                    # so a bad tool call doesn't crash your whole loop
                    print(f"Calling tool {t_name}...")
                    t_out = f(**t_args)
                except Exception as e:
                    t_out = f"Error calling tool: {e}"
            else:
                print(f"Error: tool not in tool dictionary.")
                
            # TODO 4: append the result back into `messages`
            # what role should this message have? (not 'user', not 'assistant')    
            messages.append({"role":"tool", "content":str(t_out), "name":t_name})
            
        # TODO 5: call ollama.chat(...) again with the updated `messages`
        # so the model can read the tool output(s) and write a real answer
        # save that result, append it to messages, and set it as your `res`
        final_res = ollama.chat(
            model=llm, 
            tools=[tool_search_web, tool_search_yf],
            messages=messages
        ) 

        res = final_res["message"]["content"]
        

    else:
        # TODO 6: no tool was called — the answer is already in agent_res
        res = agent_res["message"]["content"]
    print("bot->", res)
        
                

you-> nvidia financial news


Calling tool search_yf...
bot-> Here's the latest financial news related to **Nvidia (NVDA)**:

1. **AI Training Leadership**: Nvidia is positioned as a leader in the AI training race, with strong momentum in its GPU technology and data center solutions. [Source: Yahoo Finance](https://finance.yahoo.com/technology/ai/articles/nvidia-may-won-ai-training-215900389.html)

2. **Strategic Investments**: The company is investing in Nebius, a startup focused on AI infrastructure, signaling confidence in long-term growth potential. [Source: Yahoo Finance](https://finance.yahoo.com/markets/stocks/articles/fastest-growing-ai-stock-looks-170500665.html)

3. **Partnerships**: Nvidia partnered with Pinterest to enhance AI-driven shopping and search features, expanding its influence in consumer tech applications. [Source: Yahoo Finance](https://finance.yahoo.com/technology/ai/articles/pinterest-nvidia-partner-boost-ai-221729051.html)

4. **Industry Outlook**: Nvidia CEO Jensen Huang addressed concer

you-> what is apple?


bot-> Apple Inc. is a multinational technology company headquartered in Cupertino, California, founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne. It is one of the world's most valuable companies by market capitalization, known for its innovative consumer electronics, software, and services. Key offerings include:

- **Products**: iPhones, MacBooks, iPads, Apple Watches, AirPods, and the M1/M2 chips.
- **Services**: App Store, Apple Music, iCloud, Apple TV+, and Apple Pay.
- **Ecosystem**: Seamless integration across devices (e.g., iPhone + MacBook) and emphasis on design, privacy, and user experience.
- **Impact**: Revolutionized industries like personal computing, mobile phones, and digital media, while facing ongoing legal and competitive challenges.

Apple’s focus on innovation and premium branding has made it a dominant force in global technology.


you-> most recent apple product released


...

you-> search for the most recent iphone released


bot-> The **most recent iPhone models released** as of **September 2023** are the **iPhone 15 series**, which includes:

### 📱 **iPhone 15 Series** (Released: September 2023)
1. **iPhone 15**  
   - Features: USB-C port, A16 Bionic chip, improved camera system, and a redesigned look.

2. **iPhone 15 Plus**  
   - Similar to the iPhone 15 but with a larger 6.7-inch display.

3. **iPhone 15 Pro**  
   - Key Features:  
     - **A17 Pro chip** (first 3nm processor in a smartphone).  
     - **Titanium frame** for durability and lighter weight.  
     - **USB-C port** (replacing Lightning).  
     - **Advanced camera system** with improved low-light performance and 48MP main sensor.

4. **iPhone 15 Pro Max**  
   - Larger 6.9-inch display, same A17 Pro chip and titanium frame as the Pro model.

### 📱 **iPhone 14 Series** (Released: September 2022)  
Still in sale and support, but not the latest.

Apple typically releases new iPhones in **September**, so these are the newest as of late 2023

you-> use the tool search_web to search for most recent iphone


Calling tool search_web...
bot-> As of the latest search results (hypothetical scenario in **2026**), the **most recent iPhone releases** include:

### 📱 **iPhone 18 Series** (Released: 2026)
1. **iPhone 18**  
   - Standard model with USB-C, A18 chip, and improved camera features.

2. **iPhone 18 Pro**  
   - Features:  
     - **A18 Pro chip** (advanced 3nm architecture).  
     - **Titanium frame** for durability.  
     - **USB-C port** (standardized across all models).  
     - **Enhanced camera system** with improved low-light performance.

3. **iPhone 18 Pro Max**  
   - Larger 6.9-inch display, same A18 Pro chip as the Pro model.

### 📱 **iPhone Duo** (Foldable iPhone, 2026)
- **First foldable iPhone** from Apple, featuring:
  - **Dual-screen design** (avoiding the "screen crease" issue).
  - **USB-C port** for versatility.
  - **Price comparable to iPad mini**, though some speculate it could disrupt tablet sales.

### Notes:
- These details are based on hypothetical 2026 relea

you-> q
